# Buscacompis
Proyecto de Matemáticas Discretas que te recomienda estudiantes con los que formar un grupo de estudio basado un tu perfil.


In [ ]:
import pandas as pd
import networkx as nx
import itertools
import matplotlib.pyplot as plt

## 1. Cargar el dataset
Leemos el CSV. Cada estudiante tiene texto separado por `;` en `materias`, `hobbies` y `horario_disponible`.
Los convertimos en conjuntos con la función set() de Python que es la estructura matemática que necesitamos.

In [ ]:
# Carga del csv con los datos de prueba
df = pd.read_csv("../data/estudiantes.csv", encoding="latin1", skiprows=1)

# Verificación con las primeras filas
df.head()

In [ ]:
def texto_a_conjunto(texto):    #Esta función convierte las columnas con varios valores en un conjunto con estos valores
    return set(texto.split(";"))



# Creamos 3 columnas nuevas como conjuntos en vez de texto
df["materias_conjunto"] = df["materias"].apply(texto_a_conjunto)
df["hobbies_conjunto"] = df["hobbies"].apply(texto_a_conjunto)
df["horario_conjunto"] = df["horario_disponible"].apply(texto_a_conjunto)

# Revisamos cómo quedó un estudiante de ejemplo
df.loc[0, ["nombre", "materias_conjunto", "hobbies_conjunto", "horario_conjunto"]]

## 2. Similitud de Jaccard
Para dos conjuntos A y B:

$$ Jaccard(A, B) = \frac{|A \cap B|}{|A \cup B|} $$

Este cálculo nos dice cuántos elementos tienen en común, sobre el total de elementos distintos entre los dos. Osea, nos dice qué tanta similitud existe entre dos conjuntos. Da un número entre 0 y 1: un 0 significa que no tienen nada en común y un 1 significa que los conjuntos son idénticos.
Usaremos este cálculo para saber que tántas cosas tienen en común dos estudiantes entre sí y, definiendo un umbral de aceptación, decidir si pueden formar un grupo de trabajo.

In [ ]:
def jaccard(conjunto_a, conjunto_b):         # Esta función calcula la similitud de Jaccard entre dos conjuntos.
                                             #Si ambos conjuntos están vacíos, devolvemos 0 para evitar dividir por cero.
    interseccion = conjunto_a & conjunto_b   # elementos en común
    union = conjunto_a | conjunto_b          # elementos distintos entre los dos
    if len(union) == 0:
        return 0.0
    return len(interseccion) / len(union)

# Prueba con dos conjuntos inventados
prueba_a = {"Cálculo I", "Programación I", "Física I"}
prueba_b = {"Cálculo I", "Programación I", "Matemáticas Discretas I"}
print("Jaccard de prueba:", jaccard(prueba_a, prueba_b))

## 3. Ponderación de similitudes
Hacemos una suma ponderada de las similitudes ya que el horario disponible y las materias son más importantes que los hobbies a la hora de asignar un posible compañero de estudio.
Para este caso definimos un peso de 0.5 para la similitud de las materias, 0.3 para la similitud de horarios y 0.2 para la similitud de hobbies, la fórmula utilizada es la siguiente:
$$Suma = (0.5 \cdot Sim\_materias) + (0.3 \cdot Sim\_horarios) + (0.2 \cdot Sim\_hobbies)$$

In [ ]:
def suma_ponderada(estudiante_a, estudiante_b, peso_materias=0.5, peso_horario=0.3, peso_hobbies=0.2): 
#Esta función recibe dos estudianteses y devueve
#una suma ponderada de las compatibilidades de los conjuntos entre 0 y 1.
    sim_materias = jaccard(estudiante_a["materias_conjunto"], estudiante_b["materias_conjunto"])
    sim_horario = jaccard(estudiante_a["horario_conjunto"], estudiante_b["horario_conjunto"])
    sim_hobbies = jaccard(estudiante_a["hobbies_conjunto"], estudiante_b["hobbies_conjunto"])
    return (peso_materias * sim_materias) + (peso_horario * sim_horario) + (peso_hobbies * sim_hobbies)

# Probamos con dos estudiantes
estudiante_1 = df.iloc[0]
estudiante_2 = df.iloc[1]

print(f"Compatibilidad entre {estudiante_1['nombre']} y {estudiante_2['nombre']}: {suma_ponderada(estudiante_1, estudiante_2):.3f}")

In [ ]:
#Probamos la suma ponderada entre varios estudiantes para ver que los números tengan sentido
parejas_prueba = [(0, 1), (0, 5), (7, 10), (0, 19)]
for i, j in parejas_prueba:
    a = df.iloc[i]
    b = df.iloc[j]
    s = suma_ponderada(a, b)
    print(f"{a['nombre']:20s} <-> {b['nombre']:20s}  score = {s:.3f}")

## 4. Construcción del grafo
Usamos `networkx` para representar a los estudiantes como nodos y sus compatibilidades como aristas.
Sólo se creará una arista entre dos estudiantes si su `suma_ponderada` supera un umbral estipulado de una manera que el grafo no tenga ni demasiadas ni muy pocas conexiones.

In [ ]:
def construir_grafo(df, umbral):        # Esta función crea un grafo de compatibilidad entre estudiantes.
                                        # Cada nodo es un estudiante identificado por su id y cada arista representa
                                        # una compatibilidad calculada con (suma_ponderada) por encima del umbral especificado.
    G = nx.Graph()
    # Añadimos todos los estudiantes como nodos, guardamos el nombre como atributo
    for _, estudiante in df.iterrows():
        G.add_node(estudiante["id"], nombre=estudiante["nombre"])
    # Recorremos todas las parejas posibles de estudiantes sin repetir
    for i, j in itertools.combinations(df.index, 2):
        a = df.iloc[i]
        b = df.iloc[j]
        compatibilidad = suma_ponderada(a, b)
        if compatibilidad >= umbral:
            G.add_edge(a["id"], b["id"], weight=compatibilidad)
    return G

In [ ]:
# Probamos distintos umbrales para ver cuál da un grafo mejor balanceado
umbrales_prueba = [0.3, 0.5, 0.6, 0.7]
for i in umbrales_prueba:
    G = construir_grafo(df, i)
    n_nodos = G.number_of_nodes()
    n_aristas = G.number_of_edges()
    densidad = nx.density(G)
    aislados = list(nx.isolates(G))
    print(f"Umbral {i:.1f}: {n_nodos} nodos, {n_aristas} aristas, densidad = {densidad:.3f}, {len(aislados)} estudiantes sin ninguna conexión")

## 6. Dibujo del grafo
Con la pruebas definimos que el umbral de 0,5 es el más adecuado, genera un buen número de conexiones sin llegar a ser un número muy alto y los nodos aislados son poco
Creamos un función para dibujar el grafo con este umbral donde el grosor de cada arista representa qué tan compatibles son dos estudiantes.

In [ ]:
umbral = 0.5
G = construir_grafo(df, umbral)
# Posiciones de los nodos (spring_layout agrupa visualmente a los más conectados)
pos = nx.spring_layout(G, seed=42, k=1.7, iterations=100)  # seed fijo para que el layout no cambie cada vez que se corre
# Etiquetas: usamos el nombre del estudiante en vez del id
etiquetas = nx.get_node_attributes(G, "nombre")
# Grosor de las aristas proporcional al peso (compatibilidad)
pesos = [G[u][v]["weight"] * 4 for u, v in G.edges()]

plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=800)
nx.draw_networkx_labels(G, pos, labels=etiquetas, font_size=8)
nx.draw_networkx_edges(G, pos, width=pesos, alpha=0.6,connectionstyle="arc3,rad=0.15", arrows=True, arrowsize=1 )
plt.title(f"Grafo de compatibilidad entre estudiantes (umbral = {umbral})")
plt.axis("off")
plt.show()

